# Add DART-derived atmospheric perturbations to reference states

This notebook constructs 20-member atmospheric ensembles for configured reference cases and date windows. EN01 is an exact unperturbed control from the reference case. A selected 20-member DART subset defines the perturbation mean, and the first 19 rows of that subset provide perturbations for EN02-EN20. The 20th selected source member is included in the mean but omitted as an individual perturbation.

```text
2011-12-16-00000, 2011-12-21-00000, 2011-12-26-00000, 2011-12-31-00000
2012-05-16-00000, 2012-05-21-00000, 2012-05-26-00000, 2012-05-31-00000
```

For selected DART subset \(S\) and member \(s_i\in S\),

\[
\delta A_{s_i} = A_{\mathrm{DART},s_i} - \overline{A}_{\mathrm{DART},S}
\]

The output atmosphere for the first 19 selected members is

\[
A_{\mathrm{output},i+1} = A_{\mathrm{reference}} + \alpha \delta A_{s_i}, \qquad i=1,\ldots,19.
\]

Because all 20 centered anomalies sum to zero, omitting the final anomaly gives the complete output ensemble the expected mean offset

\[
\overline{A}_{\mathrm{output}} - A_{\mathrm{reference}} = -\frac{\alpha\,\delta A_{\mathrm{omitted}}}{20}.
\]

Validation checks closure against this expected offset. The selected-member CSV is paired with its date set:

```text
dec2011 -> /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/dart_subset_selection/2011-12-16-00000/selected_20_members.csv
may2012 -> /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/dart_subset_selection/2012-05-16-00000/selected_20_members.csv
```

The HYB perturbable atmospheric targets come from `DART_INIT/ATM`. The FREE_CLIM reference state is read from `F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM/archive/rest/<date>/*.eam.i.<date>.nc` and written as `eam.i` initial-condition files. Matching non-atmosphere component restart files are not modified and are linked into each date's initial-condition library.

The output is organized by date, for example:

```text
/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_dartpert20_atm_multidate/2011-12-21-00000/
/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate/2012-05-21-00000/
```

Each date directory contains the case-specific `*.EN01-EN20.<stream>.<date>.nc` files, the output-to-DART mapping, perturbation and humidity-clipping diagnostics, written-file validation, component restart links, and metadata. Root-level `all_*.csv` summaries describe only the configurations processed in the current notebook run.

In [7]:
from pathlib import Path
import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from netCDF4 import Dataset

try:
    display
except NameError:
    def display(obj):
        print(obj)


In [8]:
# -------------------------------------------------------------------
# User configuration: multi-reference perturbation libraries
# -------------------------------------------------------------------
# Optional run selectors. Use one month or reference at a time for separate runs.
# Lists preserve order, e.g. RUN_REFERENCE_NAMES = ["FREE_CLIM", "HYB"].
# Sets/strings select names but keep the configured order.
RUN_DATE_SET_NAMES = 'dec2011' #'may2012'  # None
RUN_REFERENCE_NAMES = 'FREE_CLIM' #'HYB' #'FREE_CLIM' #None

WRITE_OUTPUT_FILES = True
# Set True when rerunning after changing perturbation settings such as ALPHA or Q_MIN.
OVERWRITE_OUTPUT_FILES = True
RESUME_EXISTING_OUTPUTS = True
CREATE_COMPONENT_LINKS = True
COMPONENT_LINK_MODE = "symlink"  # "symlink" or "copy"
REQUIRE_NCOL_D_FOR_EAM_I = True
EXPECTED_EAM_I_NCOL_D_SIZE = 48602

ATM_VARS_TO_PERTURB = ["T", "U", "V", "Q", "PS"]
SELECTED_DART_MEMBER_ROOT = Path(
    "/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/"
    "dart_subset_selection"
)
CONTROL_OUTPUT_MEMBER = 1
N_SELECTED_DART_MEMBERS = 20
N_PERTURBED_OUTPUTS = 19

ALPHA = 1.0
CLIP_Q = True
Q_MIN = 0.0

# Date-specific HYB eam.i targets compatible with the DART atmospheric anomalies.
HYB_REF_ATM_DIR = Path("/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM")
# Non-atmosphere component restart sets used by the HYB initial conditions.
HYB_REF_REST_ROOT = Path(
    "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
    "20241109.v3-LR.DATM.ne30pg2_r05_IcoswISC30E3r5.pm-cpu/archive/rest"
)

FREE_CLIM_CASE_PREFIX = "F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM"
FREE_CLIM_REF_REST_ROOT = Path(
    "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
    f"{FREE_CLIM_CASE_PREFIX}/archive/rest"
)

DATE_SETS = [
    {
        "set_name": "dec2011",
        "dates": ["2011-12-16-00000", "2011-12-21-00000", "2011-12-26-00000", "2011-12-31-00000"],
        "selected_dart_member_csv": SELECTED_DART_MEMBER_ROOT / "2011-12-16-00000" / "selected_20_members.csv",
        "dart_case_prefix": "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        "dart_rest_root": Path(
            "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
            "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/rest"
        ),
        "dart_diag_root": Path(
            "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
            "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/dart_en40"
        ),
    },
    {
        "set_name": "may2012",
        "dates": ["2012-05-16-00000", "2012-05-21-00000", "2012-05-26-00000", "2012-05-31-00000"],
        "selected_dart_member_csv": SELECTED_DART_MEMBER_ROOT / "2012-05-16-00000" / "selected_20_members.csv",
        "dart_case_prefix": "DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        "dart_rest_root": Path(
            "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
            "DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/rest"
        ),
        "dart_diag_root": Path(
            "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
            "DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/dart_en40"
        ),
    },
]

REFERENCE_CASES = [
    {
        "ref_name": "HYB",
        "ref_atm_source": "hyb_eam_i",
        "ref_atm_dir": HYB_REF_ATM_DIR,
        "ref_rest_root": HYB_REF_REST_ROOT,
        "output_root": Path("/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_dartpert20_atm_multidate"),
        "out_case_prefix": "HYB_DARTPERT20",
        "out_atm_stream": "eam.i",
    },
    {
        "ref_name": "FREE_CLIM",
        "ref_atm_source": "free_clim_eam_i",
        "ref_case_prefix": FREE_CLIM_CASE_PREFIX,
        "ref_rest_root": FREE_CLIM_REF_REST_ROOT,
        "output_root": Path("/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate"),
        "out_case_prefix": "FREECLIM_DARTPERT20",
        "out_atm_stream": "eam.i",
    },
]

ENSEMBLE_MEAN_TOLERANCE = {
    "T": 1.0e-5,
    "U": 1.0e-5,
    "V": 1.0e-5,
    "Q": 1.0e-10,
    "PS": 1.0e-2,
}
PHYSICAL_LIMITS = {
    "T": (150.0, 400.0),
    "U": (-300.0, 300.0),
    "V": (-300.0, 300.0),
    "Q": (0.0, 0.1),
    "PS": (4.0e4, 1.2e5),
}
WARN_Q_CLIP_FRACTION = 1.0e-2
ENFORCE_MAX_Q_CLIP_FRACTION = False
MAX_Q_CLIP_FRACTION = 2.0e-2
DIMENSION_ALIASES = {"ncol_d": "ncol"}
GRID_COORDINATE_ATOL_DEGREES = 2.0e-5
FLOAT32_COORDINATE_RTOL = 1.0e-6
FLOAT32_COORDINATE_ATOL = 1.0e-7
COMPONENT_PATTERNS = [
    "*.elm.r.*.nc",
    "*.mosart.r.*.nc",
    "*.mpaso.rst.*.nc",
    "*.mpassi.rst.*.nc",
    "*.cpl.r.*.nc",
    "rpointer.drv",
    "rpointer.ice",
    "rpointer.lnd",
    "rpointer.ocn",
    "rpointer.rof",
]
OPTIONAL_COMPONENT_PATTERNS = {
    "*.mpaso.rst.*.nc",  # AMIP/DOCN cases do not have MPAS-O restart files.
}


In [9]:

def selected_names_or_all(requested, available, label, preserve_requested_order=False):
    available = list(available)
    available_set = set(available)
    if requested is None:
        return available
    if isinstance(requested, str):
        requested = [requested]
    requested = list(requested)
    unknown = set(requested) - available_set
    if unknown:
        raise ValueError(
            f"Unknown {label}: {sorted(unknown)}. Available: {sorted(available_set)}"
        )
    if preserve_requested_order:
        return requested
    return [name for name in available if name in set(requested)]

active_date_set_names = selected_names_or_all(
    RUN_DATE_SET_NAMES,
    [date_set["set_name"] for date_set in DATE_SETS],
    "date set names",
    preserve_requested_order=isinstance(RUN_DATE_SET_NAMES, (list, tuple)),
)
active_reference_names = selected_names_or_all(
    RUN_REFERENCE_NAMES,
    [ref_case["ref_name"] for ref_case in REFERENCE_CASES],
    "reference names",
    preserve_requested_order=isinstance(RUN_REFERENCE_NAMES, (list, tuple)),
)
DATE_SETS_BY_NAME = {date_set["set_name"]: date_set for date_set in DATE_SETS}
REFERENCE_CASES_BY_NAME = {ref_case["ref_name"]: ref_case for ref_case in REFERENCE_CASES}
DATE_SETS = [DATE_SETS_BY_NAME[name] for name in active_date_set_names]
REFERENCE_CASES = [REFERENCE_CASES_BY_NAME[name] for name in active_reference_names]
if not DATE_SETS:
    raise ValueError("No DATE_SETS selected.")
if not REFERENCE_CASES:
    raise ValueError("No REFERENCE_CASES selected.")

for ref_case in REFERENCE_CASES:
    ref_case["output_root"].mkdir(parents=True, exist_ok=True)

print("Reference cases:")
for ref_case in REFERENCE_CASES:
    print(
        ref_case["ref_name"],
        "source=", ref_case["ref_atm_source"],
        "ref_rest_root=", ref_case["ref_rest_root"],
        "output_root=", ref_case["output_root"].resolve(),
    )
for date_set in DATE_SETS:
    print(date_set["set_name"], date_set["dates"])

Reference cases:
FREE_CLIM source= free_clim_eam_i ref_rest_root= /global/cfs/cdirs/m4849/zhan391/e3sm_dart/F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM/archive/rest output_root= /global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate
dec2011 ['2011-12-16-00000', '2011-12-21-00000', '2011-12-26-00000', '2011-12-31-00000']


## 1. Load the selected DART members and define output mapping


In [10]:
def load_selected_dart_members(csv_path: Path, expected_count: int) -> list[int]:
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Selected-member CSV not found: {csv_path}. "
            "Run the 20-member DART subset-selection notebook first."
        )
    table = pd.read_csv(csv_path)
    if "member" in table.columns:
        members = table["member"].astype(int).tolist()
    elif "member_label" in table.columns:
        members = [int(str(value).upper().removeprefix("EN")) for value in table["member_label"]]
    else:
        raise ValueError("Selected-member CSV requires 'member' or 'member_label'.")
    if len(members) != len(set(members)):
        raise ValueError("Selected-member CSV contains duplicate members.")
    if len(members) != expected_count:
        raise ValueError(f"Expected {expected_count} selected members, found {len(members)}.")
    invalid = [member for member in members if member < 1 or member > 40]
    if invalid:
        raise ValueError(f"Selected DART members outside EN01-EN40: {invalid}")
    return members

perturbed_output_members = list(range(2, 2 + N_PERTURBED_OUTPUTS))
all_output_members = [CONTROL_OUTPUT_MEMBER] + perturbed_output_members
if all_output_members != list(range(1, 21)):
    raise ValueError("Final output ensemble must be EN01-EN20.")

def build_member_selection(csv_path: Path) -> dict:
    selected = load_selected_dart_members(csv_path, N_SELECTED_DART_MEMBERS)
    perturbation_sources = selected[:N_PERTURBED_OUTPUTS]
    omitted = selected[N_PERTURBED_OUTPUTS:]
    if len(omitted) != 1:
        raise ValueError("Exactly one selected DART source member must be omitted.")
    output_mapping = dict(zip(perturbed_output_members, perturbation_sources))
    member_mapping = pd.DataFrame({
        "output_member": perturbed_output_members,
        "output_member_label": [f"EN{m:02d}" for m in perturbed_output_members],
        "dart_member": perturbation_sources,
        "dart_member_label": [f"EN{m:02d}" for m in perturbation_sources],
    })
    complete_mapping = pd.concat([
        pd.DataFrame([{
            "output_member": CONTROL_OUTPUT_MEMBER,
            "output_member_label": "EN01",
            "dart_member": pd.NA,
            "dart_member_label": "CONTROL",
        }]),
        member_mapping,
    ], ignore_index=True)
    return {
        "selected_dart_member_csv": csv_path,
        "selected_dart_members": selected,
        "centering_dart_members": selected,
        "perturbation_dart_members": perturbation_sources,
        "omitted_dart_members": omitted,
        "output_to_dart_member": output_mapping,
        "complete_member_mapping": complete_mapping,
    }


## 2. Build per-date configuration table


In [11]:
def ref_atm_file(ref_case: dict, date_tag: str) -> Path:
    if ref_case["ref_atm_source"] == "hyb_eam_i":
        return ref_case["ref_atm_dir"] / f"v3.LR.ne30pg2.ERA5.eam.i.{date_tag}.nc"
    if ref_case["ref_atm_source"] == "free_clim_eam_i":
        return (
            ref_case["ref_rest_root"]
            / date_tag
            / f"{ref_case['ref_case_prefix']}.eam.i.{date_tag}.nc"
        )
    raise ValueError(f"Unknown reference atmosphere source: {ref_case['ref_atm_source']}")

def ref_rest_dir(ref_case: dict, date_tag: str) -> Path:
    return ref_case["ref_rest_root"] / date_tag

def output_dir_for(ref_case: dict, set_name: str, date_tag: str) -> Path:
    return ref_case["output_root"] / date_tag

def dart_rest_dir(date_set: dict, date_tag: str) -> Path:
    return date_set["dart_rest_root"] / date_tag

def dart_diag_dir(date_set: dict, date_tag: str) -> Path:
    return date_set["dart_diag_root"] / date_tag

def dart_rest_member_file(date_set: dict, date_tag: str, member: int) -> Path:
    prefix = date_set["dart_case_prefix"]
    filename = f"{prefix}.EN{member:02d}.eam.i.{date_tag}.nc"
    shared_path = dart_rest_dir(date_set, date_tag) / filename
    if shared_path.is_file():
        return shared_path
    case_root = date_set["dart_rest_root"].parent.parent
    return case_root / f"EN{member:02d}" / "archive" / "rest" / date_tag / filename

def dart_forecast_member_file(date_set: dict, date_tag: str, member: int) -> Path:
    prefix = date_set["dart_case_prefix"]
    return dart_diag_dir(date_set, date_tag) / f"{prefix}.eam_{member:04d}.e.forecast.{date_tag}"

def dart_forecast_mean_file(date_set: dict, date_tag: str) -> Path:
    prefix = date_set["dart_case_prefix"]
    return dart_diag_dir(date_set, date_tag) / f"{prefix}.dart.e.eam_forecast_mean.{date_tag}.nc"

def has_usable_vars(path: Path, variables) -> bool:
    if not path.is_file():
        return False
    try:
        with xr.open_dataset(path, decode_times=False) as ds:
            for var in variables:
                if var not in ds:
                    return False
                if any(size == 0 for size in ds[var].shape):
                    return False
    except Exception:
        return False
    return True

def dataset_dim_size(path: Path, dim_name: str):
    if not path.is_file():
        return None
    with xr.open_dataset(path, decode_times=False) as ds:
        return ds.sizes.get(dim_name)

def variable_horizontal_size(path: Path, variables):
    if not path.is_file():
        return None
    with xr.open_dataset(path, decode_times=False) as ds:
        for var in variables:
            if var not in ds:
                continue
            for dim in ("ncol_d", "ncol"):
                if dim in ds[var].dims:
                    return ds.sizes[dim]
    return None

def variable_horizontal_dims(path: Path, variables):
    if not path.is_file():
        return {}
    dims = {}
    with xr.open_dataset(path, decode_times=False) as ds:
        for var in variables:
            if var not in ds:
                continue
            dims[var] = tuple(dim for dim in ds[var].dims if dim in {"ncol", "ncol_d"})
    return dims

def assert_eam_i_uses_ncol_d_or_equivalent(path: Path, variables, label, expected_size=None):
    bad = {}
    with xr.open_dataset(path, decode_times=False) as ds:
        file_ncol_d = ds.sizes.get("ncol_d")
        for var in variables:
            if var not in ds:
                continue
            horizontal_dims = tuple(dim for dim in ds[var].dims if dim in {"ncol", "ncol_d"})
            if "ncol_d" in horizontal_dims:
                continue
            if "ncol" in horizontal_dims:
                var_size = ds.sizes["ncol"]
                matches_file_ncol_d = file_ncol_d is not None and var_size == file_ncol_d
                matches_expected = expected_size is not None and var_size == expected_size
                if matches_file_ncol_d or matches_expected:
                    continue
            bad[var] = horizontal_dims
    if bad:
        raise ValueError(
            f"{label} should use ncol_d for EAM IC variables, or ncol only "
            f"when ncol has the expected ncol_d size ({expected_size}). "
            f"Found {bad} in {path}"
        )

def choose_dart_source(date_set: dict, date_tag: str, selected_members) -> dict:
    rest_files = {m: dart_rest_member_file(date_set, date_tag, m) for m in selected_members}
    rest_ok = all(has_usable_vars(path, ATM_VARS_TO_PERTURB) for path in rest_files.values())
    if rest_ok:
        return {
            "source_type": "rest_eam_i",
            "member_files": rest_files,
            "mean_file": None,
            "mean_mode": "compute_from_members",
        }

    forecast_files = {m: dart_forecast_member_file(date_set, date_tag, m) for m in selected_members}
    forecast_ok = all(has_usable_vars(path, ATM_VARS_TO_PERTURB) for path in forecast_files.values())
    if forecast_ok:
        return {
            "source_type": "dart_en40_forecast",
            "member_files": forecast_files,
            "mean_file": None,
            "mean_mode": "compute_selected_subset_mean",
        }

    raise FileNotFoundError(f"No usable DART atmospheric source found for {date_tag}")

run_configs = []
for ref_case in REFERENCE_CASES:
    for date_set in DATE_SETS:
        member_selection = build_member_selection(date_set["selected_dart_member_csv"])
        for date_tag in date_set["dates"]:
            source = choose_dart_source(
                date_set, date_tag, member_selection["selected_dart_members"]
            )
            cfg = {
                "ref_name": ref_case["ref_name"],
                "ref_atm_source": ref_case["ref_atm_source"],
                "set_name": date_set["set_name"],
                "date_tag": date_tag,
                "dart_case_prefix": date_set["dart_case_prefix"],
                "dart_rest_dir": dart_rest_dir(date_set, date_tag),
                "dart_diag_dir": dart_diag_dir(date_set, date_tag),
                "dart_source_type": source["source_type"],
                "dart_member_files": source["member_files"],
                "dart_mean_file": source["mean_file"],
                "dart_mean_mode": source["mean_mode"],
                **member_selection,
                "ref_atm_file": ref_atm_file(ref_case, date_tag),
                "ref_rest_dir": ref_rest_dir(ref_case, date_tag),
                "output_root": ref_case["output_root"],
                "output_dir": output_dir_for(ref_case, date_set["set_name"], date_tag),
                "out_case_prefix": ref_case["out_case_prefix"],
                "out_atm_stream": ref_case["out_atm_stream"],
            }
            run_configs.append(cfg)

config_table = pd.DataFrame([
    {
        "ref_name": cfg["ref_name"],
        "set": cfg["set_name"],
        "date": cfg["date_tag"],
        "selected_member_csv": str(cfg["selected_dart_member_csv"]),
        "selected_members": cfg["selected_dart_members"],
        "dart_source_type": cfg["dart_source_type"],
        "dart_mean_mode": cfg["dart_mean_mode"],
        "ref_atm_exists": cfg["ref_atm_file"].is_file(),
        "ref_horizontal_dims": variable_horizontal_dims(cfg["ref_atm_file"], ATM_VARS_TO_PERTURB),
        "ref_perturbation_ncol": variable_horizontal_size(cfg["ref_atm_file"], ATM_VARS_TO_PERTURB),
        "dart_horizontal_dims": variable_horizontal_dims(next(iter(cfg["dart_member_files"].values())), ATM_VARS_TO_PERTURB),
        "dart_perturbation_ncol": variable_horizontal_size(next(iter(cfg["dart_member_files"].values())), ATM_VARS_TO_PERTURB),
        "ref_rest_exists": cfg["ref_rest_dir"].is_dir(),
        "ref_atm_file": str(cfg["ref_atm_file"]),
        "output_dir": str(cfg["output_dir"]),
    }
    for cfg in run_configs
])
display(config_table)

missing_ref = [
    f"{cfg['ref_name']} {cfg['date_tag']}"
    for cfg in run_configs
    if not cfg["ref_atm_file"].is_file() or not cfg["ref_rest_dir"].is_dir()
]
if missing_ref:
    raise FileNotFoundError("Missing reference files/directories for " + ", ".join(missing_ref))

if REQUIRE_NCOL_D_FOR_EAM_I:
    for cfg in run_configs:
        dart_size = variable_horizontal_size(
            next(iter(cfg["dart_member_files"].values())), ATM_VARS_TO_PERTURB
        )
        expected_size = EXPECTED_EAM_I_NCOL_D_SIZE or dart_size
        assert_eam_i_uses_ncol_d_or_equivalent(
            cfg["ref_atm_file"], ATM_VARS_TO_PERTURB,
            f"{cfg['ref_name']} {cfg['date_tag']} reference atmosphere",
            expected_size=expected_size,
        )
        assert_eam_i_uses_ncol_d_or_equivalent(
            next(iter(cfg["dart_member_files"].values())), ATM_VARS_TO_PERTURB,
            f"{cfg['ref_name']} {cfg['date_tag']} DART member source",
            expected_size=expected_size,
        )


,ref_name,set,date,selected_member_csv,selected_members,dart_source_type,dart_mean_mode,ref_atm_exists,ref_horizontal_dims,ref_perturbation_ncol,dart_horizontal_dims,dart_perturbation_ncol,ref_rest_exists,ref_atm_file,output_dir
0,FREE_CLIM,dec2011,2011-12-16-00000,/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_a...,"[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25,...",rest_eam_i,compute_from_members,True,"{'T': ('ncol_d',), 'U': ('ncol_d',), 'V': ('nc...",48602,"{'T': ('ncol_d',), 'U': ('ncol_d',), 'V': ('nc...",48602,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/F20T...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free...
1,FREE_CLIM,dec2011,2011-12-21-00000,/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_a...,"[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25,...",rest_eam_i,compute_from_members,True,"{'T': ('ncol_d',), 'U': ('ncol_d',), 'V': ('nc...",48602,"{'T': ('ncol_d',), 'U': ('ncol_d',), 'V': ('nc...",48602,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/F20T...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free...
2,FREE_CLIM,dec2011,2011-12-26-00000,/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_a...,"[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25,...",rest_eam_i,compute_from_members,True,"{'T': ('ncol_d',), 'U': ('ncol_d',), 'V': ('nc...",48602,"{'T': ('ncol_d',), 'U': ('ncol_d',), 'V': ('nc...",48602,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/F20T...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free...
3,FREE_CLIM,dec2011,2011-12-31-00000,/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_a...,"[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25,...",rest_eam_i,compute_from_members,True,"{'T': ('ncol_d',), 'U': ('ncol_d',), 'V': ('nc...",48602,"{'T': ('ncol_d',), 'U': ('ncol_d',), 'V': ('nc...",48602,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/F20T...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free...


## 3. Compatibility and alignment helpers


In [12]:
def inventory(path: Path, variables):
    rows = []
    with xr.open_dataset(path, decode_times=False) as ds:
        for var in variables:
            if var not in ds:
                rows.append({"variable": var, "status": "missing", "dims": "", "shape": "", "dtype": "", "units": "", "has_empty_dim": True})
            else:
                da = ds[var]
                rows.append({
                    "variable": var,
                    "status": "found",
                    "dims": str(da.dims),
                    "shape": str(da.shape),
                    "dtype": str(da.dtype),
                    "units": da.attrs.get("units", ""),
                    "has_empty_dim": any(s == 0 for s in da.shape),
                })
    return pd.DataFrame(rows)

def ref_coords_for(ref_da):
    return {dim: ref_da.coords[dim] for dim in ref_da.dims if dim in ref_da.coords}

def validate_alignment(da, ref_da, label):
    rename_dims = {
        dim: DIMENSION_ALIASES[dim]
        for dim in da.dims
        if dim in DIMENSION_ALIASES
        and DIMENSION_ALIASES[dim] in ref_da.dims
        and DIMENSION_ALIASES[dim] not in da.dims
    }
    normalized = da.rename(rename_dims)
    if set(normalized.dims) != set(ref_da.dims):
        raise ValueError(
            f"{label}: DART dims={da.dims}, REF dims={ref_da.dims}"
        )

    aligned = normalized.transpose(*ref_da.dims)
    if aligned.shape != ref_da.shape:
        raise ValueError(
            f"{label}: shape after transpose={aligned.shape}, REF={ref_da.shape}"
        )

    for dim in ref_da.dims:
        if dim not in aligned.coords or dim not in ref_da.coords:
            continue
        if aligned.sizes[dim] == 1 and ref_da.sizes[dim] == 1:
            continue
        dart_coord = np.asarray(aligned.coords[dim].values)
        ref_coord = np.asarray(ref_da.coords[dim].values)
        if dart_coord.shape != ref_coord.shape:
            raise ValueError(f"{label}: coordinate size mismatch for {dim}")
        if np.issubdtype(dart_coord.dtype, np.number) and np.issubdtype(ref_coord.dtype, np.number):
            uses_float32 = (
                np.issubdtype(dart_coord.dtype, np.floating) and dart_coord.dtype.itemsize <= 4
            ) or (
                np.issubdtype(ref_coord.dtype, np.floating) and ref_coord.dtype.itemsize <= 4
            )
            rtol = FLOAT32_COORDINATE_RTOL if uses_float32 else 1.0e-10
            atol = FLOAT32_COORDINATE_ATOL if uses_float32 else 1.0e-12
            matches = np.allclose(
                dart_coord, ref_coord, rtol=rtol, atol=atol, equal_nan=True
            )
        else:
            matches = np.array_equal(dart_coord, ref_coord)
        if not matches:
            raise ValueError(f"{label}: coordinate mismatch for {dim}")
    return aligned

def horizontal_dim(ds):
    if "ncol_d" in ds.sizes:
        return "ncol_d"
    if "ncol" in ds.sizes:
        return "ncol"
    return None

def horizontal_coord_names(ds, dim):
    if dim == "ncol_d" and "lat_d" in ds and "lon_d" in ds:
        return "lat_d", "lon_d"
    if "lat" in ds and "lon" in ds:
        return "lat", "lon"
    return None, None

def validate_horizontal_grid(dart_ds, ref_ds, label):
    dart_dim = horizontal_dim(dart_ds)
    ref_dim = horizontal_dim(ref_ds)
    if dart_dim is None or ref_dim is None:
        return
    if dart_ds.sizes[dart_dim] != ref_ds.sizes[ref_dim]:
        raise ValueError(
            f"{label}: DART {dart_dim}={dart_ds.sizes[dart_dim]} != "
            f"reference {ref_dim}={ref_ds.sizes[ref_dim]}"
        )
    dart_lat_name, dart_lon_name = horizontal_coord_names(dart_ds, dart_dim)
    ref_lat_name, ref_lon_name = horizontal_coord_names(ref_ds, ref_dim)
    required = [
        (dart_ds, dart_lat_name), (dart_ds, dart_lon_name),
        (ref_ds, ref_lat_name), (ref_ds, ref_lon_name),
    ]
    missing = [name for ds, name in required if name not in ds]
    if missing:
        raise ValueError(f"{label}: missing grid coordinates {missing}")
    dart_lat = np.asarray(dart_ds[dart_lat_name].values)
    dart_lon = np.asarray(dart_ds[dart_lon_name].values)
    ref_lat = np.asarray(ref_ds[ref_lat_name].values)
    ref_lon = np.asarray(ref_ds[ref_lon_name].values)
    if not np.allclose(
        dart_lat, ref_lat, rtol=0.0, atol=GRID_COORDINATE_ATOL_DEGREES
    ):
        raise ValueError(f"{label}: latitude grid mismatch")
    wrapped_lon_difference = (dart_lon - ref_lon + 180.0) % 360.0 - 180.0
    if not np.allclose(
        wrapped_lon_difference, 0.0, rtol=0.0, atol=GRID_COORDINATE_ATOL_DEGREES
    ):
        raise ValueError(f"{label}: longitude grid mismatch")

def align_to_ref_dims(da, ref_da):
    aligned = validate_alignment(da, ref_da, da.name or "unnamed variable")
    return aligned.assign_coords(ref_coords_for(ref_da))

def load_aligned_var(path: Path, var: str, ref_ds):
    with xr.open_dataset(path, decode_times=False, mask_and_scale=True, cache=False) as ds:
        da = ds[var].load().astype("float64")
    return align_to_ref_dims(da, ref_ds[var])

def validate_all_member_structures(cfg):
    errors = []
    with xr.open_dataset(
        cfg["ref_atm_file"], decode_times=False, mask_and_scale=True, cache=False
    ) as ref_ds:
        missing_ref = [var for var in ATM_VARS_TO_PERTURB if var not in ref_ds]
        if missing_ref:
            errors.append(f"Reference missing variables: {missing_ref}")

        for member in cfg["selected_dart_members"]:
            path = cfg["dart_member_files"][member]
            try:
                with xr.open_dataset(
                    path, decode_times=False, mask_and_scale=True, cache=False
                ) as dart_ds:
                    validate_horizontal_grid(dart_ds, ref_ds, f"EN{member:02d}")
                    for var in ATM_VARS_TO_PERTURB:
                        if var not in ref_ds or var not in dart_ds:
                            if var not in dart_ds:
                                errors.append(f"EN{member:02d} missing {var}: {path}")
                            continue
                        if not np.issubdtype(dart_ds[var].dtype, np.number):
                            errors.append(f"EN{member:02d} {var} is not numeric: {path}")
                            continue
                        try:
                            validate_alignment(
                                dart_ds[var], ref_ds[var], f"EN{member:02d} {var}"
                            )
                        except ValueError as exc:
                            errors.append(str(exc))
            except Exception as exc:
                errors.append(f"Could not inspect EN{member:02d} {path}: {exc}")

    if errors:
        raise ValueError(
            "DART member compatibility check failed:\n" + "\n".join(errors)
        )
    return list(ATM_VARS_TO_PERTURB)

def assert_finite(da, label):
    values = np.asarray(da.values)
    n_nan = int(np.isnan(values).sum())
    n_inf = int(np.isinf(values).sum())
    if n_nan or n_inf:
        raise ValueError(f"{label} contains {n_nan} NaNs and {n_inf} infinities")

def compute_member_mean(member_files, selected_members, variables, ref_file):
    mean_vars = {}
    with xr.open_dataset(ref_file, decode_times=False, mask_and_scale=True, cache=False) as ref_ds:
        for var in variables:
            print(f"Computing selected 20-member DART mean for {var}")
            running = np.zeros(ref_ds[var].shape, dtype=np.float64)
            for m in selected_members:
                da = load_aligned_var(member_files[m], var, ref_ds)
                assert_finite(da, f"EN{m:02d} {var}")
                running += da.values
            mean_vars[var] = xr.DataArray(
                running / len(selected_members),
                dims=ref_ds[var].dims,
                coords=ref_coords_for(ref_ds[var]),
                attrs=dict(ref_ds[var].attrs),
                name=var,
            )
            assert_finite(mean_vars[var], f"DART ensemble mean {var}")
    return xr.Dataset(mean_vars)

def load_dart_mean(cfg, variables):
    if cfg["dart_mean_mode"] in {
        "compute_from_members", "compute_selected_subset_mean"
    }:
        return compute_member_mean(
            cfg["dart_member_files"], cfg["selected_dart_members"],
            variables, cfg["ref_atm_file"]
        )
    raise ValueError(f"Unknown mean mode: {cfg['dart_mean_mode']}")

compat_rows = []
for cfg in run_configs:
    print(f"Preflighting all 20 selected members for {cfg['date_tag']}")
    cfg["compatible_vars"] = validate_all_member_structures(cfg)
    compat_rows.append({
        "set": cfg["set_name"],
        "date": cfg["date_tag"],
        "dart_source_type": cfg["dart_source_type"],
        "compatible_vars": ",".join(cfg["compatible_vars"]),
    })
display(pd.DataFrame(compat_rows))


Preflighting all 20 selected members for 2011-12-16-00000
Preflighting all 20 selected members for 2011-12-21-00000
Preflighting all 20 selected members for 2011-12-26-00000
Preflighting all 20 selected members for 2011-12-31-00000


,set,date,dart_source_type,compatible_vars
0,dec2011,2011-12-16-00000,rest_eam_i,"T,U,V,Q,PS"
1,dec2011,2011-12-21-00000,rest_eam_i,"T,U,V,Q,PS"
2,dec2011,2011-12-26-00000,rest_eam_i,"T,U,V,Q,PS"
3,dec2011,2011-12-31-00000,rest_eam_i,"T,U,V,Q,PS"


## 4. Diagnostics, writing, and validation functions


In [13]:
def output_atm_file(cfg, member: int) -> Path:
    return cfg["output_dir"] / f"{cfg['out_case_prefix']}.EN{member:02d}.{cfg['out_atm_stream']}.{cfg['date_tag']}.nc"

def assert_no_new_invalid(reference, candidate, label):
    ref_valid = np.isfinite(np.asarray(reference.values))
    candidate_valid = np.isfinite(np.asarray(candidate.values))
    newly_invalid = ref_valid & ~candidate_valid
    if np.any(newly_invalid):
        raise ValueError(
            f"{label} introduced {int(newly_invalid.sum())} new invalid values"
        )

def enforce_physical_limits(var, da, label):
    lower, upper = PHYSICAL_LIMITS[var]
    actual_min = float(da.min(skipna=True).item())
    actual_max = float(da.max(skipna=True).item())
    if (
        not np.isfinite(actual_min)
        or not np.isfinite(actual_max)
        or actual_min < lower
        or actual_max > upper
    ):
        raise ValueError(
            f"{label} outside broad physical limits: min={actual_min}, "
            f"max={actual_max}, allowed=({lower}, {upper})"
        )

def perturbation_diagnostics(cfg, dart_mean):
    rows = []
    with xr.open_dataset(cfg["ref_atm_file"], decode_times=False, mask_and_scale=True, cache=False) as ref_ds:
        for var in cfg["compatible_vars"]:
            member_mean_delta = []
            member_rms_delta = []
            assert_finite(dart_mean[var], f"DART ensemble mean {var}")
            for m in cfg["selected_dart_members"]:
                dart_var = load_aligned_var(cfg["dart_member_files"][m], var, ref_ds)
                assert_finite(dart_var, f"EN{m:02d} {var}")
                perturb_da = (dart_var - dart_mean[var]).load()
                assert_finite(perturb_da, f"EN{m:02d} {var} perturbation")
                perturb = perturb_da.values
                member_mean_delta.append(float(np.mean(perturb)))
                member_rms_delta.append(float(np.sqrt(np.mean(perturb ** 2))))
            rows.append({
                "ref_name": cfg["ref_name"],
                "date": cfg["date_tag"],
                "variable": var,
                "mean_of_member_mean_perturbations": float(np.mean(member_mean_delta)),
                "std_of_member_mean_perturbations": float(np.std(member_mean_delta, ddof=1)),
                "mean_member_rms_perturbation": float(np.mean(member_rms_delta)),
                "min_member_rms_perturbation": float(np.min(member_rms_delta)),
                "max_member_rms_perturbation": float(np.max(member_rms_delta)),
            })
    return pd.DataFrame(rows)

def compare_attribute_values(ref_value, out_value):
    try:
        return np.array_equal(
            np.asarray(ref_value), np.asarray(out_value), equal_nan=True
        )
    except TypeError:
        return np.array_equal(np.asarray(ref_value), np.asarray(out_value))

def validate_same_netcdf_structure(ref_file, out_file):
    errors = []
    with Dataset(ref_file, mode="r") as ref_nc, Dataset(out_file, mode="r") as out_nc:
        if ref_nc.data_model != out_nc.data_model:
            errors.append(f"Data model changed: {ref_nc.data_model} -> {out_nc.data_model}")

        ref_dims = list(ref_nc.dimensions)
        out_dims = list(out_nc.dimensions)
        if ref_dims != out_dims:
            errors.append(f"Dimension names/order changed: {ref_dims} -> {out_dims}")
        for dim_name in ref_dims:
            if dim_name not in out_nc.dimensions:
                continue
            ref_dim = ref_nc.dimensions[dim_name]
            out_dim = out_nc.dimensions[dim_name]
            if len(ref_dim) != len(out_dim):
                errors.append(f"Dimension {dim_name} size changed")
            if ref_dim.isunlimited() != out_dim.isunlimited():
                errors.append(f"Dimension {dim_name} unlimited status changed")

        ref_vars = list(ref_nc.variables)
        out_vars = list(out_nc.variables)
        if ref_vars != out_vars:
            errors.append("Variable names or ordering changed")

        ref_global_attrs = list(ref_nc.ncattrs())
        out_global_attrs = list(out_nc.ncattrs())
        if ref_global_attrs != out_global_attrs:
            errors.append("Global attribute names or ordering changed")
        for attr in ref_global_attrs:
            if attr in out_global_attrs and not compare_attribute_values(
                ref_nc.getncattr(attr), out_nc.getncattr(attr)
            ):
                errors.append(f"Global attribute {attr!r} changed")

        for var_name in ref_vars:
            if var_name not in out_nc.variables:
                continue
            ref_var = ref_nc.variables[var_name]
            out_var = out_nc.variables[var_name]
            structural_pairs = {
                "dtype": (str(ref_var.dtype), str(out_var.dtype)),
                "dimensions": (tuple(ref_var.dimensions), tuple(out_var.dimensions)),
                "shape": (tuple(ref_var.shape), tuple(out_var.shape)),
                "filters": (ref_var.filters(), out_var.filters()),
                "chunking": (ref_var.chunking(), out_var.chunking()),
                "endian": (ref_var.endian(), out_var.endian()),
                "attributes": (list(ref_var.ncattrs()), list(out_var.ncattrs())),
            }
            for property_name, (ref_value, out_value) in structural_pairs.items():
                if ref_value != out_value:
                    errors.append(f"{var_name} {property_name} changed")
            for attr in ref_var.ncattrs():
                if attr in out_var.ncattrs() and not compare_attribute_values(
                    ref_var.getncattr(attr), out_var.getncattr(attr)
                ):
                    errors.append(f"{var_name} attribute {attr!r} changed")

    if errors:
        raise RuntimeError(
            f"NetCDF structure validation failed for {out_file}:\n"
            + "\n".join(errors)
        )

def validate_unperturbed_variables_identical(ref_file, out_file, perturbed_variables):
    with Dataset(ref_file, mode="r") as ref_nc, Dataset(out_file, mode="r") as out_nc:
        for var_name in ref_nc.variables:
            if var_name in perturbed_variables:
                continue
            ref_var = ref_nc.variables[var_name]
            out_var = out_nc.variables[var_name]
            ref_var.set_auto_maskandscale(False)
            out_var.set_auto_maskandscale(False)
            if not np.array_equal(ref_var[:], out_var[:]):
                raise RuntimeError(f"Unperturbed variable {var_name} changed in {out_file}")

def report_variable_storage(ref_file, variables):
    with Dataset(ref_file, mode="r") as nc:
        for var_name in variables:
            nc_var = nc.variables[var_name]
            print(
                var_name,
                "dtype=", nc_var.dtype,
                "dimensions=", nc_var.dimensions,
                "chunking=", nc_var.chunking(),
                "filters=", nc_var.filters(),
                "scale_factor=", getattr(nc_var, "scale_factor", None),
                "add_offset=", getattr(nc_var, "add_offset", None),
                "_FillValue=", getattr(nc_var, "_FillValue", None),
            )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_control_file(cfg):
    ref_file = cfg["ref_atm_file"]
    out_file = output_atm_file(cfg, CONTROL_OUTPUT_MEMBER)
    if out_file.exists() and not OVERWRITE_OUTPUT_FILES:
        if not RESUME_EXISTING_OUTPUTS:
            raise FileExistsError(
                f"Output already exists: {out_file}. "
                "Set RESUME_EXISTING_OUTPUTS=True to validate and reuse it."
            )
        ref_checksum = sha256_file(ref_file)
        control_checksum = sha256_file(out_file)
        if control_checksum != ref_checksum:
            raise RuntimeError(
                f"Existing EN01 checksum does not match the reference: {out_file}"
            )
        print(f"Reusing checksum-verified control: {out_file}")
        return out_file, control_checksum
    tmp_file = out_file.with_suffix(out_file.suffix + ".tmp")
    if tmp_file.exists():
        tmp_file.unlink()
    try:
        shutil.copy2(ref_file, tmp_file)
        ref_checksum = sha256_file(ref_file)
        control_checksum = sha256_file(tmp_file)
        if control_checksum != ref_checksum:
            raise RuntimeError("EN01 control checksum does not match the reference.")
        tmp_file.replace(out_file)
    except Exception:
        if tmp_file.exists():
            tmp_file.unlink()
        raise
    return out_file, control_checksum

def write_member_file(cfg, output_member: int, dart_member: int, dart_mean):
    out_file = output_atm_file(cfg, output_member)
    ref_file = cfg["ref_atm_file"]
    if out_file.exists() and not OVERWRITE_OUTPUT_FILES:
        if not RESUME_EXISTING_OUTPUTS:
            raise FileExistsError(
                f"Output already exists: {out_file}. "
                "Set RESUME_EXISTING_OUTPUTS=True to validate and reuse it."
            )
        validate_same_netcdf_structure(ref_file, out_file)
        validate_unperturbed_variables_identical(
            ref_file, out_file, set(cfg["compatible_vars"])
        )
        prior_q_rows = []
        q_diag_file = cfg["output_dir"] / "q_clipping_diagnostics.csv"
        if q_diag_file.is_file() and q_diag_file.stat().st_size:
            prior_q = pd.read_csv(q_diag_file)
            if "output_member" in prior_q.columns:
                prior_q_rows = prior_q.loc[
                    prior_q["output_member"] == output_member
                ].to_dict(orient="records")
        print(f"Reusing structure-verified member: {out_file}")
        return out_file, prior_q_rows
    tmp_file = out_file.with_suffix(out_file.suffix + ".tmp")
    if tmp_file.exists():
        tmp_file.unlink()

    q_clip_rows = []
    updated_values = {}
    with xr.open_dataset(
        ref_file, decode_times=False, mask_and_scale=True, cache=False
    ) as ref_ds:
        for var in cfg["compatible_vars"]:
            ref_da = ref_ds[var].load().astype("float64")
            dart_var = load_aligned_var(cfg["dart_member_files"][dart_member], var, ref_ds)
            assert_finite(dart_var, f"DART EN{dart_member:02d} {var}")
            assert_finite(dart_mean[var], f"DART ensemble mean {var}")
            perturb = dart_var - dart_mean[var].astype("float64")
            assert_finite(perturb, f"DART EN{dart_member:02d} {var} perturbation")
            new_da = ref_da + ALPHA * perturb

            if var == "Q" and CLIP_Q:
                clip_mask = new_da < Q_MIN
                number_clipped = int(clip_mask.sum().item())
                fraction_clipped = float(clip_mask.mean().item())
                minimum_before_clip = float(new_da.min(skipna=True).item())
                maximum_q_correction = max(0.0, Q_MIN - minimum_before_clip)
                exceeds_q_clip_warning = fraction_clipped > WARN_Q_CLIP_FRACTION
                q_clip_rows.append({
                    "ref_name": cfg["ref_name"],
                    "date": cfg["date_tag"],
                    "output_member": output_member,
                    "output_member_label": f"EN{output_member:02d}",
                    "dart_member": dart_member,
                    "dart_member_label": f"EN{dart_member:02d}",
                    "number_q_clipped": number_clipped,
                    "fraction_q_clipped": fraction_clipped,
                    "minimum_q_before_clip": minimum_before_clip,
                    "maximum_q_correction": maximum_q_correction,
                    "exceeds_q_clip_warning": exceeds_q_clip_warning,
                })
                if exceeds_q_clip_warning:
                    print(
                        f"Warning: output EN{output_member:02d} from DART EN{dart_member:02d} "
                        f"Q clipping fraction {fraction_clipped:.6e} "
                        f"exceeds warning threshold {WARN_Q_CLIP_FRACTION:.6e}"
                    )
                if ENFORCE_MAX_Q_CLIP_FRACTION and fraction_clipped > MAX_Q_CLIP_FRACTION:
                    raise ValueError(
                        f"Output EN{output_member:02d} from DART EN{dart_member:02d} "
                        f"Q clipping fraction {fraction_clipped:.6e} "
                        f"exceeds hard limit {MAX_Q_CLIP_FRACTION:.6e}"
                    )
                new_da = new_da.clip(min=Q_MIN)

            if new_da.dims != ref_da.dims or new_da.shape != ref_da.shape:
                raise ValueError(
                    f"Output EN{output_member:02d} {var} changed structure: "
                    f"dims={new_da.dims}, shape={new_da.shape}"
                )
            assert_no_new_invalid(ref_da, new_da, f"output EN{output_member:02d} {var}")
            enforce_physical_limits(var, new_da, f"output EN{output_member:02d} {var}")
            updated_values[var] = np.asarray(new_da.values)

    try:
        shutil.copy2(ref_file, tmp_file)
        with Dataset(tmp_file, mode="r+") as nc:
            for var, values in updated_values.items():
                if var not in nc.variables:
                    raise KeyError(f"{var} missing from copied file {tmp_file}")
                nc_var = nc.variables[var]
                if nc_var.shape != values.shape:
                    raise ValueError(
                        f"{var}: NetCDF shape {nc_var.shape} != {values.shape}"
                    )
                nc_var[:] = values
            nc.sync()
        validate_same_netcdf_structure(ref_file, tmp_file)
        validate_unperturbed_variables_identical(
            ref_file, tmp_file, set(cfg["compatible_vars"])
        )
        tmp_file.replace(out_file)
    except Exception:
        if tmp_file.exists():
            tmp_file.unlink()
        raise
    return out_file, q_clip_rows

def validate_written_files(cfg, dart_mean):
    rows = []
    with xr.open_dataset(cfg["ref_atm_file"], decode_times=False, mask_and_scale=True, cache=False) as ref_ds:
        for var in cfg["compatible_vars"]:
            member_mean_delta = []
            member_rms_delta = []
            member_min_delta = []
            member_max_delta = []
            ens_sum = None
            for output_member in perturbed_output_members:
                with xr.open_dataset(output_atm_file(cfg, output_member), decode_times=False, mask_and_scale=True, cache=False) as out_ds:
                    delta = (out_ds[var] - ref_ds[var]).load()
                    vals = delta.values
                    assert_no_new_invalid(
                        ref_ds[var], out_ds[var],
                        f"written output EN{output_member:02d} {var}",
                    )
                    member_mean_delta.append(float(np.mean(vals)))
                    member_rms_delta.append(float(np.sqrt(np.mean(vals ** 2))))
                    member_min_delta.append(float(np.min(vals)))
                    member_max_delta.append(float(np.max(vals)))
                    ens_sum = delta.astype("float64") if ens_sum is None else ens_sum + delta.astype("float64")
            perturbed_mean_delta = ens_sum / len(perturbed_output_members)
            complete_mean_delta = ens_sum / len(all_output_members)
            omitted_member = cfg["omitted_dart_members"][0]
            omitted_var = load_aligned_var(
                cfg["dart_member_files"][omitted_member], var, ref_ds
            )
            expected_complete_mean_delta = (
                -ALPHA * (omitted_var - dart_mean[var]) / len(all_output_members)
            )
            mean_closure_error = (
                complete_mean_delta - expected_complete_mean_delta
            )
            perturbed_mean_values = perturbed_mean_delta.values
            complete_mean_values = complete_mean_delta.values
            expected_complete_mean_values = expected_complete_mean_delta.values
            mean_closure_error_values = mean_closure_error.values
            rows.append({
                "ref_name": cfg["ref_name"],
                "date": cfg["date_tag"],
                "variable": var,
                "mean_member_mean_delta": float(np.mean(member_mean_delta)),
                "mean_member_rms_delta": float(np.mean(member_rms_delta)),
                "min_delta_across_members": float(np.min(member_min_delta)),
                "max_delta_across_members": float(np.max(member_max_delta)),
                "rms_perturbed_mean_delta": float(np.sqrt(np.mean(perturbed_mean_values ** 2))),
                "max_abs_perturbed_mean_delta": float(np.max(np.abs(perturbed_mean_values))),
                "rms_complete_mean_delta": float(np.sqrt(np.mean(complete_mean_values ** 2))),
                "max_abs_complete_mean_delta": float(np.max(np.abs(complete_mean_values))),
                "rms_expected_complete_mean_delta": float(np.sqrt(np.mean(expected_complete_mean_values ** 2))),
                "max_abs_expected_complete_mean_delta": float(np.max(np.abs(expected_complete_mean_values))),
                "rms_mean_closure_error": float(np.sqrt(np.mean(mean_closure_error_values ** 2))),
                "max_abs_mean_closure_error": float(np.max(np.abs(mean_closure_error_values))),
            })
    return pd.DataFrame(rows)

def enforce_validation(validation):
    failures = []
    for row in validation.itertuples(index=False):
        tolerance = ENSEMBLE_MEAN_TOLERANCE[row.variable]
        if row.variable == "Q" and CLIP_Q:
            continue
        if row.max_abs_mean_closure_error > tolerance:
            failures.append(
                f"{row.date} {row.variable}: max_abs_mean_closure_error="
                f"{row.max_abs_mean_closure_error:.6e} > {tolerance:.6e}"
            )
    if failures:
        raise RuntimeError("Written perturbation validation failed:\n" + "\n".join(failures))

def component_files(cfg):
    files = []
    for pattern in COMPONENT_PATTERNS:
        matches = list(cfg["ref_rest_dir"].glob(pattern))
        if not matches:
            if pattern in OPTIONAL_COMPONENT_PATTERNS:
                continue
            raise FileNotFoundError(
                f"No non-atmosphere component files matching {pattern!r} "
                f"in {cfg['ref_rest_dir']}"
            )
        files.extend(matches)
    return sorted(set(files))

def maybe_link_components(cfg):
    if not CREATE_COMPONENT_LINKS:
        return []
    linked_files = []
    for src in component_files(cfg):
        out_file = cfg["output_dir"] / src.name
        if out_file.exists() or out_file.is_symlink():
            if RESUME_EXISTING_OUTPUTS and not OVERWRITE_OUTPUT_FILES:
                if (
                    COMPONENT_LINK_MODE == "symlink"
                    and out_file.is_symlink()
                    and out_file.resolve() == src.resolve()
                ):
                    linked_files.append(out_file)
                    continue
            if not OVERWRITE_OUTPUT_FILES:
                raise FileExistsError(f"Component output already exists: {out_file}")
            out_file.unlink()
        if COMPONENT_LINK_MODE == "symlink":
            out_file.symlink_to(src)
        elif COMPONENT_LINK_MODE == "copy":
            shutil.copy2(src, out_file)
        else:
            raise ValueError("COMPONENT_LINK_MODE must be 'symlink' or 'copy'.")
        linked_files.append(out_file)
    return linked_files

def write_metadata(cfg, written_files, linked_files, control_checksum):
    metadata = {
        "ref_name": cfg["ref_name"],
        "ref_atm_source": cfg["ref_atm_source"],
        "date_tag": cfg["date_tag"],
        "set_name": cfg["set_name"],
        "dart_case_prefix": cfg["dart_case_prefix"],
        "dart_rest_dir": str(cfg["dart_rest_dir"]),
        "dart_diag_dir": str(cfg["dart_diag_dir"]),
        "dart_source_type": cfg["dart_source_type"],
        "dart_mean_file": str(cfg["dart_mean_file"]) if cfg["dart_mean_file"] else None,
        "dart_mean_mode": cfg["dart_mean_mode"],
        "ref_atm_file": str(cfg["ref_atm_file"]),
        "ref_rest_dir": str(cfg["ref_rest_dir"]),
        "method": "selected-subset-centered DART perturbations",
        "selected_dart_member_csv": str(cfg["selected_dart_member_csv"]),
        "control_output_member": "EN01",
        "control_sha256": control_checksum,
        "selected_dart_members": cfg["selected_dart_members"],
        "centering_members": cfg["centering_dart_members"],
        "perturbation_source_members": cfg["perturbation_dart_members"],
        "omitted_source_member": cfg["omitted_dart_members"][0],
        "output_to_dart_member": {
            f"EN{output_member:02d}": f"EN{dart_member:02d}"
            for output_member, dart_member in cfg["output_to_dart_member"].items()
        },
        "compatible_vars": cfg["compatible_vars"],
        "alpha": ALPHA,
        "clip_q": CLIP_Q,
        "q_min": Q_MIN,
        "output_dir": str(cfg["output_dir"].resolve()),
        "out_case_prefix": cfg["out_case_prefix"],
        "out_atm_stream": cfg["out_atm_stream"],
        "write_output_files": WRITE_OUTPUT_FILES,
        "overwrite_output_files": OVERWRITE_OUTPUT_FILES,
        "resume_existing_outputs": RESUME_EXISTING_OUTPUTS,
        "ensemble_mean_tolerance": ENSEMBLE_MEAN_TOLERANCE,
        "physical_limits": PHYSICAL_LIMITS,
        "warn_q_clip_fraction": WARN_Q_CLIP_FRACTION,
        "enforce_max_q_clip_fraction": ENFORCE_MAX_Q_CLIP_FRACTION,
        "max_q_clip_fraction": MAX_Q_CLIP_FRACTION,
        "created_atmosphere_files": [str(p) for p in written_files],
        "create_component_links": CREATE_COMPONENT_LINKS,
        "component_link_mode": COMPONENT_LINK_MODE,
        "created_component_links": [str(p) for p in linked_files],
        "formula": "A_output = A_REF + alpha*(A_DART_selected - mean(A_DART_selected_20_member_subset))",
        "expected_complete_mean_offset": "-alpha*(A_DART_omitted - mean(A_DART_selected_20_member_subset))/20",
    }
    metadata_file = cfg["output_dir"] / "dart_perturbation_metadata.json"
    with open(metadata_file, "w") as f:
        json.dump(metadata, f, indent=2)
    return metadata_file


## 5. Generate the multi-date HYB perturbation library


In [14]:
all_diag = []
all_validation = []
all_q_clip = []
metadata_files = []

for cfg in run_configs:
    print("\n" + "=" * 80)
    print(f"Processing {cfg['ref_name']} {cfg['set_name']} {cfg['date_tag']} using {cfg['dart_source_type']}")
    cfg["output_dir"].mkdir(parents=True, exist_ok=True)
    report_variable_storage(cfg["ref_atm_file"], cfg["compatible_vars"])
    print("Selected-member CSV:", cfg["selected_dart_member_csv"])
    print("DART centering members:", cfg["centering_dart_members"])
    print("Omitted DART source member:", cfg["omitted_dart_members"][0])
    display(cfg["complete_member_mapping"])
    cfg["complete_member_mapping"].to_csv(
        cfg["output_dir"] / "output_to_dart_member_mapping.csv", index=False
    )

    dart_mean = load_dart_mean(cfg, cfg["compatible_vars"])

    diag = perturbation_diagnostics(cfg, dart_mean)
    display(diag)
    diag.to_csv(cfg["output_dir"] / "perturbation_diagnostics.csv", index=False)
    all_diag.append(diag)

    written_files = []
    q_clip_rows = []
    control_checksum = None
    if WRITE_OUTPUT_FILES:
        print(f"Writing {cfg['date_tag']} EN01 exact control")
        control_file, control_checksum = write_control_file(cfg)
        written_files.append(control_file)

        for output_member, dart_member in cfg["output_to_dart_member"].items():
            print(
                f"Writing {cfg['date_tag']} EN{output_member:02d} "
                f"from DART EN{dart_member:02d}"
            )
            out_file, member_q_clip_rows = write_member_file(
                cfg, output_member, dart_member, dart_mean
            )
            written_files.append(out_file)
            q_clip_rows.extend(member_q_clip_rows)
        if len(written_files) != len(all_output_members):
            raise RuntimeError(
                f"Expected {len(all_output_members)} outputs, wrote {len(written_files)}."
            )
        print(f"Wrote {len(written_files)} atmosphere files to {cfg['output_dir']}")

        q_clip = pd.DataFrame(q_clip_rows)
        display(q_clip)
        q_clip.to_csv(cfg["output_dir"] / "q_clipping_diagnostics.csv", index=False)
        if not q_clip.empty:
            all_q_clip.append(q_clip)

        validation = validate_written_files(cfg, dart_mean)
        validation["ensemble_mean_tolerance"] = validation["variable"].map(
            ENSEMBLE_MEAN_TOLERANCE
        )
        validation["ensemble_mean_tolerance_enforced"] = ~((
            validation["variable"] == "Q"
        ) & CLIP_Q)
        validation["passes_tolerance"] = (
            ~validation["ensemble_mean_tolerance_enforced"]
            | (
                validation["max_abs_mean_closure_error"]
                <= validation["ensemble_mean_tolerance"]
            )
        )
        display(validation)
        validation.to_csv(cfg["output_dir"] / "written_file_validation.csv", index=False)
        enforce_validation(validation)
        all_validation.append(validation)
    else:
        print("WRITE_OUTPUT_FILES=False; no files written.")

    linked_files = maybe_link_components(cfg)
    if linked_files:
        print(f"Created {len(linked_files)} component {COMPONENT_LINK_MODE}s/copies.")
    else:
        print("No component links/copies created.")

    metadata_file = write_metadata(
        cfg, written_files, linked_files, control_checksum
    )
    metadata_files.append(metadata_file)
    print("Metadata written to:", metadata_file)

if all_diag:
    all_diag_df = pd.concat(all_diag, ignore_index=True)
    display(all_diag_df)
    for output_root in sorted({cfg["output_root"] for cfg in run_configs}, key=str):
        case_diag = all_diag_df.loc[
            all_diag_df["ref_name"].isin([
                cfg["ref_name"] for cfg in run_configs if cfg["output_root"] == output_root
            ])
        ]
        case_diag.to_csv(output_root / "all_perturbation_diagnostics.csv", index=False)

if all_validation:
    all_validation_df = pd.concat(all_validation, ignore_index=True)
    display(all_validation_df)
    for output_root in sorted({cfg["output_root"] for cfg in run_configs}, key=str):
        case_validation = all_validation_df.loc[
            all_validation_df["ref_name"].isin([
                cfg["ref_name"] for cfg in run_configs if cfg["output_root"] == output_root
            ])
        ]
        case_validation.to_csv(output_root / "all_written_file_validation.csv", index=False)

if all_q_clip:
    all_q_clip_df = pd.concat(all_q_clip, ignore_index=True)
    for output_root in sorted({cfg["output_root"] for cfg in run_configs}, key=str):
        case_q_clip = all_q_clip_df.loc[
            all_q_clip_df["ref_name"].isin([
                cfg["ref_name"] for cfg in run_configs if cfg["output_root"] == output_root
            ])
        ]
        case_q_clip.to_csv(output_root / "all_q_clipping_diagnostics.csv", index=False)

print("Metadata files:")
for path in metadata_files:
    print(path)



Processing FREE_CLIM dec2011 2011-12-16-00000 using rest_eam_i
T dtype= float64 dimensions= ('time', 'lev', 'ncol_d') chunking= None filters= None scale_factor= None add_offset= None _FillValue= None
U dtype= float64 dimensions= ('time', 'lev', 'ncol_d') chunking= None filters= None scale_factor= None add_offset= None _FillValue= None
V dtype= float64 dimensions= ('time', 'lev', 'ncol_d') chunking= None filters= None scale_factor= None add_offset= None _FillValue= None
Q dtype= float64 dimensions= ('time', 'lev', 'ncol_d') chunking= None filters= None scale_factor= None add_offset= None _FillValue= None
PS dtype= float64 dimensions= ('time', 'ncol_d') chunking= None filters= None scale_factor= None add_offset= None _FillValue= None
Selected-member CSV: /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/dart_subset_selection/2011-12-16-00000/selected_20_members.csv
DART centering members: [1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25, 26, 28, 31, 35, 36, 37, 40]
Omitted 

,output_member,output_member_label,dart_member,dart_member_label
0,1,EN01,<NA>,CONTROL
1,2,EN02,1,EN01
2,3,EN03,3,EN03
3,4,EN04,4,EN04
4,5,EN05,7,EN07
5,6,EN06,8,EN08
6,7,EN07,9,EN09
7,8,EN08,10,EN10
8,9,EN09,14,EN14
9,10,EN10,17,EN17


Computing selected 20-member DART mean for T
Computing selected 20-member DART mean for U
Computing selected 20-member DART mean for V
Computing selected 20-member DART mean for Q
Computing selected 20-member DART mean for PS


,ref_name,date,variable,mean_of_member_mean_perturbations,std_of_member_mean_perturbations,mean_member_rms_perturbation,min_member_rms_perturbation,max_member_rms_perturbation
0,FREE_CLIM,2011-12-16-00000,T,1.309716e-17,0.081981,1.423773,1.366500,1.507740
1,FREE_CLIM,2011-12-16-00000,U,0.000000e+00,0.071874,3.816014,3.651504,4.040798
2,FREE_CLIM,2011-12-16-00000,V,8.456777e-19,0.035780,3.765881,3.637172,3.986581
3,FREE_CLIM,2011-12-16-00000,Q,-1.270549e-22,0.000004,0.000323,0.000312,0.000332
4,FREE_CLIM,2011-12-16-00000,PS,-3.632650e-14,72.843675,150.800345,125.036263,278.319974


Writing 2011-12-16-00000 EN01 exact control
Writing 2011-12-16-00000 EN02 from DART EN01
Writing 2011-12-16-00000 EN03 from DART EN03
Writing 2011-12-16-00000 EN04 from DART EN04
Writing 2011-12-16-00000 EN05 from DART EN07
Writing 2011-12-16-00000 EN06 from DART EN08
Writing 2011-12-16-00000 EN07 from DART EN09
Writing 2011-12-16-00000 EN08 from DART EN10
Writing 2011-12-16-00000 EN09 from DART EN14
Writing 2011-12-16-00000 EN10 from DART EN17
Writing 2011-12-16-00000 EN11 from DART EN18
Writing 2011-12-16-00000 EN12 from DART EN20
Writing 2011-12-16-00000 EN13 from DART EN21
Writing 2011-12-16-00000 EN14 from DART EN25
Writing 2011-12-16-00000 EN15 from DART EN26
Writing 2011-12-16-00000 EN16 from DART EN28
Writing 2011-12-16-00000 EN17 from DART EN31
Writing 2011-12-16-00000 EN18 from DART EN35
Writing 2011-12-16-00000 EN19 from DART EN36
Writing 2011-12-16-00000 EN20 from DART EN37
Wrote 20 atmosphere files to /global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multi

...

,ref_name,date,variable,mean_member_mean_delta,mean_member_rms_delta,min_delta_across_members,max_delta_across_members,rms_perturbed_mean_delta,max_abs_perturbed_mean_delta,rms_complete_mean_delta,max_abs_complete_mean_delta,rms_expected_complete_mean_delta,max_abs_expected_complete_mean_delta,rms_mean_closure_error,max_abs_mean_closure_error,ensemble_mean_tolerance,ensemble_mean_tolerance_enforced,passes_tolerance
0,FREE_CLIM,2011-12-31-00000,T,0.002027,1.926372,-57.801550,56.392564,0.102529,2.196312,0.097402,2.086496,0.097402,2.086496,3.414924e-14,1.961101e-13,1.000000e-05,True,True
1,FREE_CLIM,2011-12-31-00000,U,-0.003684,5.325406,-122.737993,153.904816,0.284922,6.040224,0.270676,5.738213,0.270676,5.738213,2.925306e-15,5.262457e-14,1.000000e-05,True,True
2,FREE_CLIM,2011-12-31-00000,V,-0.003241,4.954629,-110.568921,152.438379,0.265992,5.224140,0.252693,4.962933,0.252693,4.962933,1.449510e-15,4.440892e-14,1.000000e-05,True,True
3,FREE_CLIM,2011-12-31-00000,Q,0.000002,0.000344,-0.010739,0.010035,0.000024,0.001151,0.000023,0.001094,0.000018,0.000420,1.518032e-05,8.841960e-04,1.000000e-10,False,True
4,FREE_CLIM,2011-12-31-00000,PS,4.027091,234.990509,-1686.882990,1401.350130,8.556648,67.728343,8.128816,64.341926,8.128816,64.341926,1.203833e-11,4.802203e-11,1.000000e-02,True,True


Created 9 component symlinks/copies.
Metadata written to: /global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate/2011-12-31-00000/dart_perturbation_metadata.json


,ref_name,date,variable,mean_of_member_mean_perturbations,std_of_member_mean_perturbations,mean_member_rms_perturbation,min_member_rms_perturbation,max_member_rms_perturbation
0,FREE_CLIM,2011-12-16-00000,T,1.309716e-17,0.081981,1.423773,1.366500,1.507740
1,FREE_CLIM,2011-12-16-00000,U,0.000000e+00,0.071874,3.816014,3.651504,4.040798
2,FREE_CLIM,2011-12-16-00000,V,8.456777e-19,0.035780,3.765881,3.637172,3.986581
3,FREE_CLIM,2011-12-16-00000,Q,-1.270549e-22,0.000004,0.000323,0.000312,0.000332
4,FREE_CLIM,2011-12-16-00000,PS,-3.632650e-14,72.843675,150.800345,125.036263,278.319974
5,FREE_CLIM,2011-12-21-00000,T,1.387779e-17,0.142292,1.579652,1.494434,1.734152
6,FREE_CLIM,2011-12-21-00000,U,2.081668e-18,0.092896,4.319897,4.014767,4.969424
7,FREE_CLIM,2011-12-21-00000,V,2.602085e-18,0.041269,4.132760,3.952287,4.568639
8,FREE_CLIM,2011-12-21-00000,Q,-3.811648e-22,0.000008,0.000353,0.000341,0.000360
9,FREE_CLIM,2011-12-21-00000,PS,7.460699e-15,173.323591,202.720433,137.074514,585.254773


,ref_name,date,variable,mean_member_mean_delta,mean_member_rms_delta,min_delta_across_members,max_delta_across_members,rms_perturbed_mean_delta,max_abs_perturbed_mean_delta,rms_complete_mean_delta,max_abs_complete_mean_delta,rms_expected_complete_mean_delta,max_abs_expected_complete_mean_delta,rms_mean_closure_error,max_abs_mean_closure_error,ensemble_mean_tolerance,ensemble_mean_tolerance_enforced,passes_tolerance
0,FREE_CLIM,2011-12-16-00000,T,0.000523,1.422327,-40.455712,47.431959,0.076381,1.273199,0.072562,1.209539,0.072562,1.209539,3.414049e-14,1.847411e-13,1.000000e-05,True,True
1,FREE_CLIM,2011-12-16-00000,U,0.002681,3.812765,-135.232581,119.815822,0.204092,4.852140,0.193887,4.609533,0.193887,4.609533,3.013713e-15,7.460699e-14,1.000000e-05,True,True
2,FREE_CLIM,2011-12-16-00000,V,-0.000099,3.764356,-113.124142,117.960428,0.199729,4.603060,0.189743,4.372907,0.189743,4.372907,1.325047e-15,3.558265e-14,1.000000e-05,True,True
3,FREE_CLIM,2011-12-16-00000,Q,0.000001,0.000319,-0.012798,0.014339,0.000021,0.001299,0.000020,0.001234,0.000016,0.000737,1.224413e-05,1.319726e-03,1.000000e-10,False,True
4,FREE_CLIM,2011-12-16-00000,PS,0.737390,151.355984,-2045.484474,1403.884412,7.381221,64.454820,7.012160,61.232079,7.012160,61.232079,1.202722e-11,5.602452e-11,1.000000e-02,True,True
5,FREE_CLIM,2011-12-21-00000,T,0.001915,1.578136,-32.401116,30.071973,0.084656,1.537078,0.080423,1.460224,0.080423,1.460224,3.413465e-14,2.017934e-13,1.000000e-05,True,True
6,FREE_CLIM,2011-12-21-00000,U,0.005031,4.315132,-129.046951,119.482105,0.232128,4.494369,0.220521,4.269651,0.220521,4.269651,3.053135e-15,7.532863e-14,1.000000e-05,True,True
7,FREE_CLIM,2011-12-21-00000,V,0.001106,4.137652,-111.028161,107.256377,0.212622,5.303308,0.201991,5.038143,0.201991,5.038143,1.444967e-15,4.263256e-14,1.000000e-05,True,True
8,FREE_CLIM,2011-12-21-00000,Q,0.000001,0.000349,-0.011690,0.011004,0.000024,0.001148,0.000022,0.001090,0.000017,0.000463,1.473840e-05,1.362334e-03,1.000000e-10,False,True
9,FREE_CLIM,2011-12-21-00000,PS,2.223750,205.508217,-1257.518995,1271.647122,7.881712,51.530317,7.487627,48.953802,7.487627,48.953802,1.207049e-11,4.802136e-11,1.000000e-02,True,True


Metadata files:
/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate/2011-12-16-00000/dart_perturbation_metadata.json
/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate/2011-12-21-00000/dart_perturbation_metadata.json
/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate/2011-12-26-00000/dart_perturbation_metadata.json
/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate/2011-12-31-00000/dart_perturbation_metadata.json
